# Compression-aware baseline for PlantCLEF 2026

This notebook provides an end-to-end baseline for training/fine-tuning on compressed 512x512 inputs and generating inference outputs with the same preprocessing path.

In [ ]:
import os
from pathlib import Path

# Ensure we are in the project root if running from the notebooks directory
if Path.cwd().name == "notebooks":
    os.chdir("..")
print(f"Current working directory: {os.getcwd()}")

In [ ]:
from pathlib import Path
import mlflow
import pandas as pd
import timm
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from tqdm.auto import tqdm
from sklearn.model_selection import train_test_split
import kornia

from src.config.compression import CompressionConfig
from src.config.mlflow_init import init_mlflow
from src.data.compression import build_normalize_transform, preprocess_pil_image
from src.models.compression import CompressAICodec

In [ ]:
cfg = CompressionConfig.from_env()
cfg.validate()
cfg.ensure_output_dirs()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(cfg)
print(f"device={device}")

In [ ]:
init_mlflow()
mlflow.set_experiment(cfg.mlflow_experiment_name)

## Data loading and compressed training dataset

In [ ]:
from sklearn.model_selection import train_test_split

train_df_full = pd.read_csv(cfg.train_metadata_path, sep=";", dtype={"partner": str})
if cfg.max_train_samples > 0:
    train_df_full = train_df_full.sample(
        n=min(cfg.max_train_samples, len(train_df_full)), random_state=42
    )

# Create class mapping from the full sample
species_ids = sorted(train_df_full["species_id"].astype(int).unique())
class_to_idx = {species_id: idx for idx, species_id in enumerate(species_ids)}
num_classes = len(class_to_idx)

# Validation split - Handle classes with only 1 member for stratified split
counts = train_df_full["species_id"].value_counts()
single_member_classes = counts[counts < 2].index.tolist()

if single_member_classes:
    df_multi = train_df_full[~train_df_full["species_id"].isin(single_member_classes)]
    df_single = train_df_full[train_df_full["species_id"].isin(single_member_classes)]
    
    if not df_multi.empty:
        try:
            train_df_strat, val_df = train_test_split(
                df_multi, test_size=0.1, random_state=42, stratify=df_multi["species_id"]
            )
            train_df = pd.concat([train_df_strat, df_single])
        except ValueError as e:
            print(f"Stratified split failed: {e}. Falling back to random split.")
            train_df, val_df = train_test_split(train_df_full, test_size=0.1, random_state=42)
    else:
        train_df = df_single
        val_df = pd.DataFrame(columns=train_df.columns)
else:
    try:
        train_df, val_df = train_test_split(
            train_df_full, test_size=0.1, random_state=42, stratify=train_df_full["species_id"]
        )
    except ValueError as e:
        print(f"Stratified split failed: {e}. Falling back to random split.")
        train_df, val_df = train_test_split(train_df_full, test_size=0.1, random_state=42)

print(f"training samples={len(train_df)} validation samples={len(val_df)} classes={num_classes}")

In [ ]:
class CompressedTrainingDataset(Dataset):
    def __init__(self, df, cfg, class_to_idx, codec, transform=None):
        self.df = df.reset_index(drop=True)
        self.cfg = cfg
        self.class_to_idx = class_to_idx
        self.codec = codec
        self.transform = transform
        self.normalize = build_normalize_transform()

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        species_id = int(row["species_id"])
        if "image_id" in row.index:
            image_name = f"{row['image_id']}.jpg"
        elif "image_name" in row.index:
            image_name = str(row["image_name"])
        else:
            raise KeyError("Metadata must contain 'image_id' or 'image_name'.")
        candidate_paths = [
            Path(self.cfg.train_image_root) / str(species_id) / image_name,
            Path(self.cfg.train_image_root) / image_name,
        ]
        if "organ" in row.index and pd.notna(row["organ"]):
            candidate_paths.append(
                Path(self.cfg.train_image_root) / str(row["organ"]) / image_name
            )
        image_path = next((p for p in candidate_paths if p.exists()), None)
        if image_path is None:
            raise FileNotFoundError(
                f"Image not found for row index {idx}: {image_name}"
            )
        image = Image.open(image_path).convert("RGB")
        tensor = preprocess_pil_image(
            image,
            image_size=self.cfg.image_size,
            jpeg_normalize_enabled=self.cfg.jpeg_normalize,
            jpeg_quality=self.cfg.jpeg_quality,
        )
        # The codec takes a batch or CHW tensor and returns x_hat in [0, 1]
        tensor = self.codec.encode_decode_with_cache(
            image_path=image_path,
            input_tensor=tensor,
            cache_dir=self.cfg.compressed_cache_dir,
        ).squeeze(0)
        
        # Apply augmentations if provided
        if self.transform:
            tensor = self.transform(tensor)
            
        # Always normalize after compression and augmentations
        tensor = self.normalize(tensor)
        
        label = self.class_to_idx[species_id]
        return tensor, label

In [ ]:
compression_device = cfg.compression_device
if compression_device.startswith("cuda"):
    compression_device = f"cuda:{cfg.compression_gpu_id}"
    if not torch.cuda.is_available():
        compression_device = "cpu"

loader_workers = cfg.num_workers
if (
    cfg.compression_enabled
    and compression_device.startswith("cuda")
    and loader_workers > 0
):
    print(
        "Forcing num_workers=0 to avoid CUDA re-init in DataLoader workers during compression."
    )
    loader_workers = 0

codec = CompressAICodec(
    enabled=cfg.compression_enabled,
    model_name=cfg.compressai_model,
    quality=cfg.compressai_quality,
    pretrained=cfg.compression_pretrained,
    device=compression_device,
)

train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),
])

train_dataset = CompressedTrainingDataset(train_df, cfg, class_to_idx, codec, transform=train_transform)
val_dataset = CompressedTrainingDataset(val_df, cfg, class_to_idx, codec, transform=None)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.train_batch_size,
    shuffle=True,
    num_workers=loader_workers,
    pin_memory=torch.cuda.is_available(),
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.train_batch_size,
    shuffle=False,
    num_workers=loader_workers,
    pin_memory=torch.cuda.is_available(),
)
print(f"train_batches={len(train_loader)} val_batches={len(val_loader)}")

## Fine-tuning loop (compressed inputs)

In [ ]:
model = timm.create_model(
    cfg.backbone_name,
    pretrained=True,
    num_classes=num_classes,
    img_size=cfg.image_size,
).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=cfg.learning_rate)

with mlflow.start_run(run_name="compression-baseline-train"):
    mlflow.log_params(
        {
            "image_size": cfg.image_size,
            "jpeg_normalize": cfg.jpeg_normalize,
            "jpeg_quality": cfg.jpeg_quality,
            "compression_enabled": cfg.compression_enabled,
            "compressai_model": cfg.compressai_model,
            "compressai_quality": cfg.compressai_quality,
            "backbone": cfg.backbone_name,
            "batch_size": cfg.train_batch_size,
            "epochs": cfg.epochs,
            "learning_rate": cfg.learning_rate,
        }
    )

    model_save_path = cfg.output_dir / "best_model_compressed.pth"
    best_val_loss = float("inf")

    for epoch in range(cfg.epochs):
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0

        pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} [Train]")
        for images, labels in pbar:
            images = images.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item() * images.size(0)
            preds = logits.argmax(dim=1)
            train_correct += (preds == labels).sum().item()
            train_total += images.size(0)
            
            pbar.set_postfix({"loss": loss.item(), "acc": train_correct / train_total})

        epoch_train_loss = train_loss / max(1, train_total)
        epoch_train_acc = train_correct / max(1, train_total)
        mlflow.log_metric("train/loss", epoch_train_loss, step=epoch)
        mlflow.log_metric("train/acc", epoch_train_acc, step=epoch)

        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0

        pbar_val = tqdm(val_loader, desc=f"Epoch {epoch+1}/{cfg.epochs} [Val]")
        with torch.no_grad():
            for images, labels in pbar_val:
                images = images.to(device)
                labels = labels.to(device)

                logits = model(images)
                loss = criterion(logits, labels)

                val_loss += loss.item() * images.size(0)
                preds = logits.argmax(dim=1)
                val_correct += (preds == labels).sum().item()
                val_total += images.size(0)
                
                pbar_val.set_postfix({"loss": loss.item(), "acc": val_correct / val_total})

        epoch_val_loss = val_loss / max(1, val_total)
        epoch_val_acc = val_correct / max(1, val_total)
        mlflow.log_metric("val/loss", epoch_val_loss, step=epoch)
        mlflow.log_metric("val/acc", epoch_val_acc, step=epoch)
        
        print(f"Epoch {epoch}: Train Loss={epoch_train_loss:.4f} Acc={epoch_train_acc:.4f} | Val Loss={epoch_val_loss:.4f} Acc={epoch_val_acc:.4f}")

        if epoch_val_loss < best_val_loss:
            best_val_loss = epoch_val_loss
            torch.save(model.state_dict(), model_save_path)
            print(f"Saved best model to {model_save_path} with val_loss={best_val_loss:.4f}")
    
    mlflow.log_artifact(str(model_save_path))

## Tiled Inference

We now perform inference on the quadrat test images using a tiling approach. Each quadrat is split into patches, each patch is passed through the compression codec, and the model predicts species scores. Predictions are then aggregated across all patches of a quadrat.

In [ ]:
import csv
import os
from kornia.contrib import extract_tensor_patches, compute_padding
from torch.amp import autocast

# Load best model
model.load_state_dict(torch.load(model_save_path))
model.eval()
print(f"Loaded best model from {model_save_path}")

# Class mapping from index back to species_id
idx_to_class = {idx: species_id for species_id, idx in class_to_idx.items()}

test_df = pd.read_csv(cfg.test_csv_path, sep=";")
test_image_dir = Path(cfg.test_image_root)

patch_size = cfg.image_size
stride = patch_size // 2
normalize = build_normalize_transform()

image_predictions = {}

with torch.no_grad():
    for idx, row in tqdm(test_df.iterrows(), total=len(test_df), desc="Quadrats"):
        quadrat_id = str(row.iloc[0]) # usually quadrat_id
        image_path = test_image_dir / f"{quadrat_id}.jpg"
        if not image_path.exists():
            # Try with other extensions or subdirs if necessary
            continue
            
        img = Image.open(image_path).convert("RGB")
        img_tensor = transforms.ToTensor()(img).unsqueeze(0) # [1, 3, H, W]
        
        h, w = img_tensor.shape[-2:]
        pad = compute_padding(original_size=(h, w), window_size=patch_size, stride=stride)
        patches = extract_tensor_patches(img_tensor, patch_size, stride, padding=pad) # [1, N, 3, PS, PS]
        patches = patches.squeeze(0) # [N, 3, PS, PS]
        
        quadrat_results = {}
        
        # Process patches in batches
        patch_batch_size = 32
        for i in range(0, patches.size(0), patch_batch_size):
            batch = patches[i : i + patch_batch_size]
            
            # Apply compression to each patch in the batch if enabled
            if cfg.compression_enabled:
                # Note: For speed, we use encode_decode_tensor instead of with_cache here
                # as caching thousands of patches might be slow/large.
                batch = codec.encode_decode_tensor(batch)
            
            batch = normalize(batch.to(device))
            
            with autocast(device.type):
                logits = model(batch)
                probs = torch.softmax(logits, dim=1)
                
                top_probs, top_indices = torch.topk(probs, cfg.top_k_tile)
                
                for p_idx in range(len(top_probs)):
                    for k in range(cfg.top_k_tile):
                        score = top_probs[p_idx, k].item()
                        cls_idx = top_indices[p_idx, k].item()
                        
                        if score > cfg.min_score:
                            species_id = idx_to_class[cls_idx]
                            if species_id not in quadrat_results or score > quadrat_results[species_id]:
                                quadrat_results[species_id] = score
        
        image_predictions[quadrat_id] = sorted(list(quadrat_results.keys()))

# Create submission
submission = pd.DataFrame(
    list(image_predictions.items()), columns=["quadrat_id", "species_ids"]
)
submission["species_ids"] = submission["species_ids"].apply(str)
submission.to_csv(cfg.submission_path, index=False, quoting=csv.QUOTE_ALL)
print(f"Saved submission with {len(submission)} rows to {cfg.submission_path}")